## Ingesting Data into Delta Lake

In [0]:
%sql
-- Set the default catalog and schema
USE CATALOG azuredatabricks;
USE SCHEMA default;

In [0]:
spark.catalog.setCurrentCatalog("azuredatabricks");
spark.catalog.setCurrentDatabase("default");

spark.catalog.listTables("default");

In [0]:
# View or List the files in my volume
spark.sql(
    " LIST '/Volumes/azuredatabricks/default/myfiles'"
).display()

## B. Delta Lake Ingestion Techniques

**Objective:** Create a Delta table from the `employees.csv` file using various methods.

- CREATE TABLE AS (CTAS)  
- UPLOAD UI (User Interface)  
- COPY INTO  
- AUTOLOADER (Overview only, outside the scope of this module)

## 1. CREATE TABLE (CTAS)

1. Create a table from the `employees.csv` file using the **CREATE TABLE AS** statement similar to the previous demonstration.  
   Run the query and confirm that the `current_employees_ctas` table was successfully created.

In [0]:
%sql
SELECT *
FROM csv.`/Volumes/azuredatabricks/default/myfiles`

In [0]:
%sql
SELECT *
FROM csv.`/Volumes/azuredatabricks/default/myfiles`

In [0]:
%sql
-- Drop table if already exists for demonstration purposes
DROP TABLE IF EXISTS current_employees_ctas;
CREATE TABLE current_employees_ctas AS
SELECT
ID,
Firstname,
Country,
Role
FROM read_files(
    '/Volumes/azuredatabricks/default/myfiles',
    format => 'csv',
    header => true,
    inferSchema => true
);

SELECT * FROM current_employees_ctas;

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
-- ALTER TABLE employees RENAME TO current_employees_ui;
SELECT * FROM current_employees_ui;

## 3. COPY INTO

Create a table from the `employees.csv` file using the **COPY INTO** statement.

The **COPY INTO** statement incrementally loads data from a file location into a Delta table. This is a retriable and idempotent operation. Files in the source location that have already been loaded are skipped. This is true even if the files have been modified since they were loaded.

1. Create an empty table named `current_employees_copyinto` and define the column data types.

> **NOTE:** You can also create an empty table with no columns and evolve the schema with `COPY INTO`.

In [0]:
%sql
DROP TABLE IF EXISTS current_employeesinto;

CREATE TABLE current_employeesinto(
    ID INT,
    Firstname STRING,
    Country STRING,
    Role STRING
);

In [0]:
%sql
DROP TABLE current_employeeinto;

2. Use the COPY INTO statement to load all files from the myfiles volume (currently only the employees.csv file exists) using the path provided by Unity Catalog. Confirm that the data is loaded into the current_employees_copyinto table.

Confirm the following:

- num_affected_rows is 4  
- num_inserted_rows is 4  
- num_skipped_correct_files is 0

In [0]:
spark.sql('''
          COPY INTO current_employeesinto
          FROM '/Volumes/azuredatabricks/default/myfiles'
          FILEFORMAT = CSV
          FORMAT_OPTIONS(
            'header' = 'true',
            'inferSchema' = 'true'
          )
  ''').display()

In [0]:
%sql
SELECT * FROM current_employeesinto;

4. Run the COPY INTO statement again and confirm that it did not re-add the data from the volume that was already loaded. Remember, COPY INTO is a retryable and idempotent operation — files in the source location that have already been loaded are skipped.

Confirm the following:

- num_affected_rows is 0  
- num_inserted_rows is 0  
- num_skipped_correct_files is 0

In [0]:
spark.sql('''
          COPY INTO current_employeesinto
          FROM '/Volumes/azuredatabricks/default/myfiles'
          FILEFORMAT = CSV
          FORMAT_OPTIONS(
            'header' = 'true',
            'inferSchema' = 'true'
          )
  ''').display()

In [0]:
files = dbutils.fs.ls('/Volumes/azuredatabricks/default/myfiles')
display(files)
                      

In [0]:
%sql
SELECT *
FROM csv.`/Volumes/azuredatabricks/default/myfiles/employees2.csv`

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/azuredatabricks/default/myfiles/employees2.csv',
  format => 'csv',
  header => true,
  inferschema => true
  )

In [0]:
spark.sql('''
          COPY INTO current_employeesinto
          FROM '/Volumes/azuredatabricks/default/myfiles'
          FILEFORMAT = CSV
          FORMAT_OPTIONS(
            'header' = 'true',
            'inferSchema' = 'true'
          )
  ''').display()

In [0]:
%sql
SELECT * FROM current_employeesinto;

In [0]:
%sql
DESCRIBE HISTORY current_employeesinto;

In [0]:
%sql
DROP TABLE IF EXISTS current_employees_ctas;
DROP TABLE IF EXISTS current_employees_ui;
DROP TABLE IF EXISTS current_employees_copyinto;
DROP TABLE IF EXISTS current_employeesinto;
DROP VIEW IF EXISTS vw_current_employees;
SHOW TABLES;

In [0]:
dbutils.fs.rm('/Volumes/azuredatabricks/default/myfiles/employees2.csv');